# 📝 Лабораторная работа 7. Текст, токены и Embeddings

Цель: пройти путь от обычной строки Python до Tensor с Embedding-векторами.


# 1. Импорт библиотек

In [ ]:
import re

import torch
import torch.nn as nn

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)

# 2. Маленький текстовый Dataset

In [ ]:
sentences = [
    "я люблю python",
    "я изучаю нейросети",
    "python помогает изучать ии",
    "нейросети работают с тензорами",
]

for sentence in sentences:
    print(sentence)

# 3. Простейший Tokenizer

In [ ]:
def simple_tokenize(text):
    text = text.lower()
    return re.findall(r"[а-яёa-z0-9]+", text)


for sentence in sentences:
    print(simple_tokenize(sentence))

# 4. Строим Vocabulary

In [ ]:
SPECIAL_TOKENS = ["<PAD>", "<UNK>"]

all_tokens = []

for sentence in sentences:
    all_tokens.extend(simple_tokenize(sentence))

unique_tokens = sorted(set(all_tokens))

vocabulary = SPECIAL_TOKENS + unique_tokens

token_to_id = {
    token: index
    for index, token in enumerate(vocabulary)
}

id_to_token = {
    index: token
    for token, index in token_to_id.items()
}

print("Vocabulary size:", len(vocabulary))

for token, token_id in token_to_id.items():
    print(token_id, "→", token)

# 5. Кодируем текст в Token IDs

In [ ]:
PAD_ID = token_to_id["<PAD>"]
UNK_ID = token_to_id["<UNK>"]


def encode(text):
    return [
        token_to_id.get(token, UNK_ID)
        for token in simple_tokenize(text)
    ]


example = "я люблю python"

print("Text:", example)
print("Tokens:", simple_tokenize(example))
print("IDs:", encode(example))

# 6. Неизвестное слово

In [ ]:
unknown_example = "я люблю transformer"

print("Tokens:", simple_tokenize(unknown_example))
print("IDs:", encode(unknown_example))
print("<UNK> ID:", UNK_ID)

# 7. Decoding

In [ ]:
def decode(token_ids):
    return [
        id_to_token[token_id]
        for token_id in token_ids
    ]


ids = encode(example)

print(ids)
print(decode(ids))

# 8. Padding

In [ ]:
encoded_sentences = [
    encode(sentence)
    for sentence in sentences
]

max_length = max(
    len(ids)
    for ids in encoded_sentences
)

print("Max length:", max_length)


def pad_sequence(token_ids, length):
    return token_ids + [PAD_ID] * (length - len(token_ids))


padded = [
    pad_sequence(ids, max_length)
    for ids in encoded_sentences
]

for row in padded:
    print(row)

# 9. Batch Tensor

In [ ]:
token_batch = torch.tensor(
    padded,
    dtype=torch.long,
)

print(token_batch)
print("Shape:", token_batch.shape)

# 10. Attention Mask

In [ ]:
attention_mask = (
    token_batch != PAD_ID
).long()

print("Token IDs:")
print(token_batch)

print("\nAttention Mask:")
print(attention_mask)

# 11. Создаём nn.Embedding

In [ ]:
VOCAB_SIZE = len(vocabulary)
EMBEDDING_DIM = 8

embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    padding_idx=PAD_ID,
)

print(embedding)
print("Embedding Matrix:", embedding.weight.shape)

# 12. Embedding одного предложения

In [ ]:
single_ids = torch.tensor(
    encode("я люблю python"),
    dtype=torch.long,
)

vectors = embedding(single_ids)

print("Input IDs shape:", single_ids.shape)
print("Output shape:", vectors.shape)
print(vectors)

# 13. Embedding Batch

In [ ]:
embedded_batch = embedding(token_batch)

print("Token Batch:", token_batch.shape)
print("Embedded Batch:", embedded_batch.shape)

# 14. Вектор конкретного Token

In [ ]:
token = "python"
token_id = token_to_id[token]

vector = embedding.weight[token_id]

print("Token:", token)
print("Token ID:", token_id)
print("Vector:")
print(vector)

# 15. Padding Vector

In [ ]:
print("PAD ID:", PAD_ID)
print("PAD vector:")
print(embedding.weight[PAD_ID])

# 16. Embeddings являются обучаемыми параметрами

In [ ]:
for name, parameter in embedding.named_parameters():
    print(
        name,
        parameter.shape,
        "requires_grad =",
        parameter.requires_grad,
    )

# 17. Минимальный backward через Embedding

In [ ]:
toy_embedding = nn.Embedding(
    num_embeddings=10,
    embedding_dim=4,
)

token_ids = torch.tensor([2, 3, 4])

output = toy_embedding(token_ids)
target = torch.zeros_like(output)

loss = ((output - target) ** 2).mean()

toy_embedding.zero_grad()
loss.backward()

print("Loss:", loss.item())
print(
    "Gradient shape:",
    toy_embedding.weight.grad.shape,
)

# 18. Косинусное сходство — только как демонстрация

In [ ]:
cosine = nn.CosineSimilarity(dim=0)

python_vector = embedding.weight[
    token_to_id["python"]
]

neural_vector = embedding.weight[
    token_to_id["нейросети"]
]

similarity = cosine(
    python_vector,
    neural_vector,
)

print("Similarity:", similarity.item())

Сейчас Embeddings случайные.

Поэтому это число **не является настоящей мерой смысловой близости**. Семантика появляется после обучения.


# 19. Главный Shape перед Transformer

In [ ]:
print(
    "[Batch, Sequence] =",
    token_batch.shape,
)

print(
    "[Batch, Sequence, Embedding] =",
    embedded_batch.shape,
)

# 20. 📌 Что нужно запомнить

```text
Text
↓
Tokenizer
↓
Token IDs
↓
Padding
↓
Attention Mask
↓
Embedding
↓
[Batch, Sequence, Embedding Dim]
```


# 21. 🧩 Эксперименты

Попробуй:

- добавить новые предложения;
- изменить Vocabulary;
- изменить `EMBEDDING_DIM`;
- посмотреть вектор каждого Token;
- добавить `<BOS>` и `<EOS>`;
- вручную сделать Batch из предложений разной длины.


# 22. ➡️ Следующая глава

# Глава 8. Transformer и Attention
